# Credit Card Fraud Detection

**Mariama Sanyang** · Data Science Portfolio  

---

## Problem statement

Given 284,807 anonymised credit card transactions made over two days (September 2013, European cardholders), identify which 492 are fraudulent.  
The class imbalance is extreme: **~0.17% fraud rate**.  

This notebook walks through the full detection pipeline:
1. EDA and the class-imbalance problem  
2. Why accuracy is a terrible metric here  
3. Proper evaluation design (stratified splits, no leakage)  
4. Baseline Logistic Regression  
5. Improved XGBoost with Optuna tuning  
6. Threshold tuning and cost analysis  
7. Feature importance and limitations  

All heavy computation is in `src/`; this notebook imports and visualises.

In [ ]:
import sys, os
# Add project root to path so src.* imports work
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

from src.config import FIGURES_DIR, RANDOM_SEED
from src.data import load_raw, split, fraud_rate

print('Imports OK')

## 1. Load data

In [ ]:
df = load_raw()
print(f'Shape: {df.shape}')
print(f'Fraud rate: {df["Class"].mean()*100:.4f}%')
df.head(3)

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print('\nClass distribution:')
print(df['Class'].value_counts())
print('\nAmount statistics by class:')
df.groupby('Class')['Amount'].describe()

## 2. The class-imbalance problem

The dataset is severely imbalanced: 284,315 legitimate transactions vs 492 fraudulent ones (0.172%).  
This creates a fundamental metric problem.

In [ ]:
img = mpimg.imread(str(FIGURES_DIR / 'class_distribution.png'))
fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

### Why accuracy fails here

A model that **always predicts "not fraud"** achieves ~99.83% accuracy while catching **zero** fraudulent transactions.  
This is the _accuracy paradox_ for imbalanced classification.

In [ ]:
from sklearn.dummy import DummyClassifier
from src.evaluate import compute_metrics

X_train, X_val, X_test, y_train, y_val, y_test = split(df)

dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
dummy.fit(X_train, y_train)

dummy_acc  = (dummy.predict(X_test) == y_test).mean()
dummy_rec  = 0.0  # always predicts 0, so no fraud ever caught

print(f'DummyClassifier (always "not fraud"):')
print(f'  Accuracy = {dummy_acc*100:.2f}%')
print(f'  Recall   = {dummy_rec*100:.2f}%  ← catches ZERO fraud')
print()
print('Takeaway: accuracy is not a useful metric here.')
print('We need Precision-Recall AUC (PR-AUC) as our primary metric.')

## 3. Evaluation design

To prevent data leakage:
- **Stratified 60/20/20 train/val/test split** with fixed seed. Test set is touched once at the end.  
- Scalers fit only on training data, inside sklearn `Pipeline`s.  
- Stratified K-fold CV (k=5) on train for model comparison, scored by PR-AUC.  
- Imbalance handled via `class_weight='balanced'` (LR) and `scale_pos_weight` (XGBoost).  
- Threshold selected on the **validation set only**, then applied unchanged to test.

In [ ]:
print('Split sizes:')
print(f'  Train: {len(X_train):,} rows  (fraud rate: {fraud_rate(y_train)*100:.4f}%)')
print(f'  Val:   {len(X_val):,} rows  (fraud rate: {fraud_rate(y_val)*100:.4f}%)')
print(f'  Test:  {len(X_test):,} rows  (fraud rate: {fraud_rate(y_test)*100:.4f}%)')

## 4. EDA: Amount and Time

In [ ]:
img = mpimg.imread(str(FIGURES_DIR / 'amount_time_by_class.png'))
fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

print('Observations:')
print('- Fraudulent transactions tend to be smaller amounts (median ~$9 vs ~$22 for legitimate).')
print('- Both classes appear throughout the two-day time window.')
print('- The Amount distribution is very right-skewed for both classes.')

## 5. Baseline: Logistic Regression

In [ ]:
results_path = FIGURES_DIR.parent / 'results.json'
with open(results_path) as f:
    results = json.load(f)

lr = results['logistic']
print(f'Logistic Regression (class_weight="balanced"):')
print(f'  CV PR-AUC = {lr["cv_pr_auc_mean"]:.4f} ± {lr["cv_pr_auc_std"]:.4f}')
print(f'  Test PR-AUC   = {lr["test_pr_auc"]:.4f}')
print(f'  Test ROC-AUC  = {lr["test_roc_auc"]:.4f}')
print(f'  Test Precision = {lr["test_precision"]:.4f}')
print(f'  Test Recall    = {lr["test_recall"]:.4f}')
print(f'  Test F1        = {lr["test_f1"]:.4f}')

## 6. Improved model: XGBoost with Optuna tuning

XGBoost handles imbalance via `scale_pos_weight` and is tuned with 50 Optuna trials.  
Early stopping on the validation set prevents overfitting.

In [ ]:
xgb = results['xgboost']
print(f'XGBoost (Optuna-tuned):')
print(f'  CV PR-AUC = {xgb["cv_pr_auc_mean"]:.4f} ± {xgb["cv_pr_auc_std"]:.4f}')
print(f'  Test PR-AUC   = {xgb["test_pr_auc"]:.4f}')
print(f'  Test ROC-AUC  = {xgb["test_roc_auc"]:.4f}')
print(f'  Test Precision = {xgb["test_precision"]:.4f}')
print(f'  Test Recall    = {xgb["test_recall"]:.4f}')
print(f'  Test F1        = {xgb["test_f1"]:.4f}')
print(f'\nBest Optuna params: {xgb["best_params"]}')

### Model comparison

In [ ]:
comparison = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'CV PR-AUC': f"{lr['cv_pr_auc_mean']:.4f} ± {lr['cv_pr_auc_std']:.4f}",
        'Test PR-AUC': f"{lr['test_pr_auc']:.4f}",
        'Test ROC-AUC': f"{lr['test_roc_auc']:.4f}",
        'Precision': f"{lr['test_precision']:.4f}",
        'Recall': f"{lr['test_recall']:.4f}",
        'F1': f"{lr['test_f1']:.4f}",
    },
    {
        'Model': 'XGBoost (tuned)',
        'CV PR-AUC': f"{xgb['cv_pr_auc_mean']:.4f} ± {xgb['cv_pr_auc_std']:.4f}",
        'Test PR-AUC': f"{xgb['test_pr_auc']:.4f}",
        'Test ROC-AUC': f"{xgb['test_roc_auc']:.4f}",
        'Precision': f"{xgb['test_precision']:.4f}",
        'Recall': f"{xgb['test_recall']:.4f}",
        'F1': f"{xgb['test_f1']:.4f}",
    },
])
comparison.set_index('Model', inplace=True)
comparison

In [ ]:
img = mpimg.imread(str(FIGURES_DIR / 'pr_curves.png'))
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(img)
ax.axis('off')
plt.title('Precision-Recall curves (test set)')
plt.tight_layout()
plt.show()

print('Note on ROC-AUC vs PR-AUC for imbalanced problems:')
print('ROC-AUC is optimistic because it uses true-negative rate (TNR) in its x-axis.')
print('With 284K legitimate transactions, a model can easily maintain high TNR even')
print('with many false positives — making ROC-AUC flatter than the actual challenge.')
print('PR-AUC focuses on the minority (fraud) class and is the right metric here.')

## 7. Threshold tuning

The default 0.5 threshold is rarely optimal. We choose the threshold on the **validation set**  
and apply it unchanged to test to avoid inflating test metrics.

In [ ]:
img = mpimg.imread(str(FIGURES_DIR / 'precision_recall_vs_threshold.png'))
fig, ax = plt.subplots(figsize=(8, 5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
img = mpimg.imread(str(FIGURES_DIR / 'pr_curve_operating_points.png'))
fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f'Chosen threshold (max F1 on val): {xgb["chosen_threshold"]:.4f}')

### Operating point trade-offs

In [ ]:
ops = results['operating_points']
rows = []
for key, op in ops.items():
    row = {
        'Operating point': op['name'],
        'Threshold': f"{op['threshold']:.3f}",
        'Precision (val)': f"{op['precision']:.4f}",
        'Recall (val)': f"{op['recall']:.4f}",
        'F1 (val)': f"{op['f1']:.4f}",
    }
    if 'total_cost' in op:
        row['Total cost (€)'] = f"{op['total_cost']:,.0f}"
    rows.append(row)

pd.DataFrame(rows).set_index('Operating point')

## 8. Final test-set results at chosen threshold

In [ ]:
img = mpimg.imread(str(FIGURES_DIR / 'confusion_matrix.png'))
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

print(f'At threshold = {xgb["chosen_threshold"]:.4f}:')
print(f'  TP = {xgb["test_tp"]}  FP = {xgb["test_fp"]}')
print(f'  FN = {xgb["test_fn"]}  TN = {xgb["test_tn"]}')

## 9. Feature importance

> **Important caveat**: V1–V28 are PCA-transformed. The original features are anonymised  
> for privacy. Feature importance tells us which PCA components are discriminative,  
> but we cannot map them back to meaningful business variables.

In [ ]:
img = mpimg.imread(str(FIGURES_DIR / 'feature_importance.png'))
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 10. Limitations

1. **Anonymised features**: V1–V28 are PCA components; we cannot interpret what drives fraud in business terms.
2. **Short time window**: only 48 hours of data. No concept drift or seasonality testing.
3. **Single dataset**: results may not generalise to other card networks or geographies.
4. **Static model**: no online learning or drift detection; fraud patterns evolve.
5. **Cost assumptions are illustrative**: the €10 review cost and amount-based fraud cost are configurable assumptions, not estimates from a real operations team.
6. **No temporal validation**: a stricter setup would use time-based splits (train on day 1, test on day 2) to simulate deployment.

## 11. Conclusions

- XGBoost with Optuna tuning substantially outperforms the Logistic Regression baseline on PR-AUC.
- Threshold tuning lets us pick the right precision/recall trade-off for the deployment context.
- The cost-based threshold provides a framework for real-world threshold selection when business costs are known.
- PR-AUC is the right primary metric; accuracy and ROC-AUC both flatter the model on this imbalanced dataset.